In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from finagent.config.universe import universe
from finagent.config.company_matching import find_title_candidates, match_company_name

assert len(universe) == 51
assert len({info["ticker"] for info in universe.values()}) == 51
tickers = [info["ticker"] for info in universe.values()]

In [ ]:
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://theedgemalaysia.com/"

PAGE_URL = (
    "https://theedgemalaysia.com/"
    "flash-categories/Companies%20in%20the%20news"
)

API_URL = (
    "https://theedgemalaysia.com/"
    "api/loadMoreFlashCategories"
)

headers = {
    "User-Agent": ("Mozilla/5.0")
}

response = requests.get(
    PAGE_URL,
    headers=headers,
    timeout=20
)

print(response.status_code)
print(response.url)
print(len(response.text))

In [ ]:
print(response.text[:500])

In [ ]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

print(soup.title)

In [ ]:
with open("page.html", "w", encoding="utf-8") as f:
    f.write(soup.prettify())


In [ ]:
import json

next_data = soup.find(
    "script",
    id="__NEXT_DATA__"
)

data = json.loads(next_data.string)

with open("HTML brut.txt", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)


In [ ]:
page_props = data["props"]["pageProps"]

initial_articles = page_props["corporateData"]

total_records = page_props["totalRecords"]


In [ ]:
import time

all_articles = initial_articles.copy()


offset = len(initial_articles)


while offset < total_records:

    params = {
        "offset": offset,
        "flash": "Companies in the news"
    }

    response = requests.get(
        API_URL,
        params=params,
        headers=headers,
        timeout=20
    )

    response.raise_for_status()

    data = response.json()

    articles = data["results"]


    if not articles:
        print("No more articles returned.")
        break


    all_articles.extend(articles)


    print(
        f"Offset {offset} -> "
        f"{len(articles)} articles received | "
        f"Total collected: {len(all_articles)}/{total_records}"
    )
    offset += len(articles)


    time.sleep(0.2)


print("\nFinished.")
print("Expected:", total_records)
print("Collected:", len(all_articles))

In [ ]:
unique_articles = {}

for article in all_articles:
    unique_articles[article["nid"]] = article

all_articles = list(
    unique_articles.values()
)

print(
    "Unique articles:",
    len(all_articles)
)

print(all_articles[0])

In [ ]:
from finagent.config.company_matching import find_title_candidates, match_company_name
from finagent.news.scraping import extract_article_blocks


In [ ]:
useful_articles = []
for article in all_articles:
    matched_companies = list(find_title_candidates(article.get("title", "")))
    if matched_companies:
        article["matched_companies"] = matched_companies
        useful_articles.append(article)
print("Articles with candidate companies:", len(useful_articles))

In [ ]:
all_company_blocks = []
rejected_blocks = []
for i, article in enumerate(useful_articles):
    print(f"Article {i + 1}/{len(useful_articles)}")
    article_url = BASE_URL + article["alias"]
    response = requests.get(article_url, headers=headers, timeout=20)
    response.raise_for_status()
    extraction = extract_article_blocks(response.text, article["nid"])
    represented = set()
    for idx, text in enumerate(extraction["paragraphs"]):
        company = match_company_name(text)
        if company is None:
            rejected_blocks.append({"nid": article["nid"], "info_index": idx,
                                    "reason": "uncertain_or_outside_universe_subject",
                                    "text_preview": text[:160]})
            continue
        if company not in article["matched_companies"] or company in represented:
            continue
        all_company_blocks.append({
            "company": company, "ticker": universe[company]["ticker"], "text": text,
            "created": article["created"], "nid": article["nid"],
            "article_title": article["title"], "article_url": article_url, "info_index": idx,
            "extraction_format": extraction["extraction_format"],
            **extraction["block_origins"][idx],
        })
        represented.add(company)
    time.sleep(0.2)
print("Rejected uncertain/outside-universe blocks:", len(rejected_blocks))
display(pd.DataFrame(rejected_blocks).head(10))

In [ ]:
df_news = pd.DataFrame(
    all_company_blocks
)

df_news["datetime"] = pd.to_datetime(
    df_news["created"],
    unit="ms",
    utc=True
)

df_news["datetime_my"] = (
    df_news["datetime"]
    .dt.tz_convert("Asia/Kuala_Lumpur")
)

df_news["date"] = (
    df_news["datetime_my"]
    .dt.date
)

df_news.to_csv("../data/preprocessed_data/news_data.csv",index=False)

In [ ]:
print(
    "Total company blocks:",
    len(all_company_blocks)
)

print(
    "\nBlocks by company:"
)

print(
    df_news["company"].value_counts()
)

In [ ]:
represented_articles = df_news["nid"].nunique()

print(
    "\nUseful articles:",
    len(useful_articles)
)

print(
    "Articles represented:",
    represented_articles
)

print(
    "Total company blocks:",
    len(df_news)
)

In [ ]:
represented_pairs = {
    (block["nid"], block["company"])
    for block in all_company_blocks
}

missing_matches = []

for article in useful_articles:

    for company in article["matched_companies"]:

        if (
            article["nid"],
            company
        ) not in represented_pairs:

            missing_matches.append({
                "nid": article["nid"],
                "company": company,
                "title": article["title"],
                "alias": article["alias"]
            })


print(
    "Company matches not found in article body:",
    len(missing_matches)
)

In [ ]:
for item in missing_matches[:20]:

    print(
        "\nCompany:",
        item["company"]
    )

    print(
        "Title:",
        item["title"]
    )

    print(
        "URL:",
        BASE_URL + item["alias"]
    )

    print("-" * 100)